# 🧪 DL-PM25-BJ — Smoke Test End-to-End trên Google Colab (CPU < 5 phút)

Kiểm tra thông suốt 100% toàn bộ pipeline (`download_data.py` -> `preprocess.py` -> `training_loop.py` -> `fine_tune.py` -> `run_ablation.py` -> `exp_backup.py`) bằng tệp cấu hình nhẹ `configs/smoke_test.yaml` (2 trạm, 1 epoch, 2 batch/epoch, 4 mô hình) trước khi chạy GPU chính thức.
- **Thư mục đầu ra cách ly:** `outputs_test/`, `mlruns_test/`, `data/preprocess_test/`, `backups_test/`
- **Google Drive Folder ID:** `1AaKHRC-_WUQ5eZSdY9nRcINfQkfcefJL`

## Cell 1 — Kiểm tra Môi trường CPU & Mount Google Drive

In [ ]:
import os
import time
from pathlib import Path
import torch

SMOKE_START = time.time()
GDRIVE_FOLDER_ID = "1AaKHRC-_WUQ5eZSdY9nRcINfQkfcefJL"
os.environ["GDRIVE_BACKUP_FOLDER_ID"] = GDRIVE_FOLDER_ID

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_TEST_DIR = Path("/content/drive/MyDrive/DL_PM25_BJ_SmokeTest")
    DRIVE_TEST_DIR.mkdir(parents=True, exist_ok=True)
    print(f"[DRIVE] Đã mount Drive tại {DRIVE_TEST_DIR}")
except Exception as exc:
    DRIVE_TEST_DIR = Path("/tmp/DL_PM25_BJ_SmokeTest")
    DRIVE_TEST_DIR.mkdir(parents=True, exist_ok=True)
    print(f"[INFO] Không chạy trên Colab ({exc}); dùng thư mục tạm {DRIVE_TEST_DIR}")

print(f"[ENV] PyTorch={torch.__version__} | CUDA={torch.cuda.is_available()} | GDRIVE_FOLDER_ID={GDRIVE_FOLDER_ID}")

## Cell 2 — Đồng bộ Mã nguồn & Cài đặt Phụ thuộc

In [ ]:
import os
import time
from pathlib import Path

t0 = time.time()
REPO_URL = "https://github.com/Duy-DS/Deep_Learning_Group.git"
BRANCH_NAME = "fix/BPM25_T0305_wp03-training-audit"
WORK_ROOT = Path("/content/beijing-pm25-deeplearning")

if not WORK_ROOT.exists():
    !git clone -b {BRANCH_NAME} {REPO_URL} {WORK_ROOT}
else:
    !git -C {WORK_ROOT} pull origin {BRANCH_NAME}

PROJECT_DIR = WORK_ROOT / "course_project"
if not PROJECT_DIR.exists():
    PROJECT_DIR = WORK_ROOT
os.chdir(PROJECT_DIR)
os.environ["MLFLOW_TRACKING_URI"] = str(PROJECT_DIR / "mlruns_test")
print(f"[CWD] {os.getcwd()}")

!grep -v '^torch' requirements.txt > /tmp/req_no_torch.txt
!pip install -q -r /tmp/req_no_torch.txt mlflow google-api-python-client google-auth-httplib2
print(f"[TIME] Cell 2 hoàn tất sau {time.time() - t0:.2f}s")

## Cell 3 — Kiểm thử Tầng Dữ liệu (`download_data.py` & `preprocess.py`)

In [ ]:
import os
import time
from pathlib import Path
import pandas as pd

if os.path.exists("/content/beijing-pm25-deeplearning/course_project"):
    os.chdir("/content/beijing-pm25-deeplearning/course_project")
t0 = time.time()

!python scripts/download_data.py --output-dir data/raw
!python -m src.data_pipeline.preprocess --config configs/smoke_test.yaml

for name in ["train.csv", "train_dev.csv", "dev.csv", "test.csv"]:
    fp = Path("data/preprocess_test") / name
    assert fp.exists(), f"Thiếu tệp {fp}"
    df = pd.read_csv(fp)
    assert df.isna().sum().sum() == 0, f"Phát hiện NaN trong {fp}"
    print(f"[VERIFIED] {fp}: shape={df.shape}, stations={df['station'].nunique()}")
print(f"[TIME] Cell 3 hoàn tất sau {time.time() - t0:.2f}s")

## Cell 4 — Kiểm thử Toàn tuyến Huấn luyện, Fine-tune & Ablation (`CPU < 5 phút`)

In [ ]:
import os
import time
from pathlib import Path

if os.path.exists("/content/beijing-pm25-deeplearning/course_project"):
    os.chdir("/content/beijing-pm25-deeplearning/course_project")
os.environ["MLFLOW_TRACKING_URI"] = str(Path.cwd() / "mlruns_test")

t1 = time.time()
!python training_loop.py --config configs/smoke_test.yaml
print(f"[TIME] training_loop.py: {time.time() - t1:.2f}s")

t2 = time.time()
!python fine_tune.py --config configs/smoke_test.yaml
print(f"[TIME] fine_tune.py: {time.time() - t2:.2f}s")

t3 = time.time()
!python run_ablation.py --config configs/smoke_test.yaml
print(f"[TIME] run_ablation.py: {time.time() - t3:.2f}s")

## Cell 5 — Kiểm thử Xuất CSV MLflow & Backup Drive + Dọn dẹp

In [ ]:
import os
import shutil
import time
from pathlib import Path
import pandas as pd

if os.path.exists("/content/beijing-pm25-deeplearning/course_project"):
    os.chdir("/content/beijing-pm25-deeplearning/course_project")
t0 = time.time()

!python -m src.utils.exp_backup --mlruns-dir mlruns_test --output-dir outputs_test --backup-dir backups_test --prefix dl_pm25_bj_SMOKE_TEST --include-checkpoints --upload-gdrive

csv_path = Path("outputs_test/mlflow_runs_export.csv")
assert csv_path.exists(), "Thiếu mlflow_runs_export.csv"
df_log = pd.read_csv(csv_path)
expected_cols = ["Start Time", "Duration", "Run ID", "Name", "Source Name", "Status"]
assert all(c in df_log.columns for c in expected_cols), f"Thiếu cột chuẩn Raw_MLflow_Logs: {df_log.columns.tolist()}"
print(f"[VERIFIED CSV] {len(df_log)} MLflow runs | Cột chuẩn khớp 100%")

latest_zip = sorted(Path("backups_test").glob("dl_pm25_bj_SMOKE_TEST_*.zip"))[-1]
if DRIVE_TEST_DIR.exists():
    shutil.copy2(latest_zip, DRIVE_TEST_DIR / latest_zip.name)
    shutil.copy2(csv_path, DRIVE_TEST_DIR / csv_path.name)
    print(f"[DRIVE TEST] Đã đồng bộ {latest_zip.name} tới {DRIVE_TEST_DIR}")

print(f"[TIME] Cell 5: {time.time() - t0:.2f}s | TỔNG THỜI GIAN SMOKE TEST: {time.time() - SMOKE_START:.2f}s")